<a href="https://colab.research.google.com/github/PMayank16/Dance-Project/blob/main/ANN_fashion_mnist_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()


In [ ]:
# Fill NaN values in the DataFrame with 0
df = df.fillna(0)

In [ ]:
#Create a 4*4 grid images
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('First 16 images', fontsize=16)

# plot the first 16 images from the dataset
for i, ax in enumerate(axes.flat):
    img = df.iloc[i, 1:].values.reshape(28,28)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"label: {df.iloc[i, 0]}") #show the label

plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout to fix the title
plt.show()

In [ ]:
#train test split
X = df.iloc[:,1:].values
Y = df.iloc[:,0].values

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
X

In [ ]:
# Create custom dataset
class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels =torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]


In [ ]:
# Crete Train dataset obj
train_dataset = CustomDataset(X_train, Y_train)


In [ ]:
# Crete test dataset obj
test_dataset = CustomDataset(X_test, Y_test)

In [ ]:
# Create train and test loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Define NN class
class MyNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64, 10)
    )
  def forward(self, x):
    return self.model(x)

In [ ]:
# Set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [ ]:
# instatiate model
model = MyNN(X_train.shape[1])

# Shift GPU
model.to(device)

# loss Function
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [ ]:
for epoch in range(epochs):

  total_epochs_loss = 0

  for batch_features, batch_labels in train_loader:

    # move to Gpu
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grade
    optimizer.step()


    total_epochs_loss = total_epochs_loss + loss.item()

avg_loss = total_epochs_loss/len(train_loader)
print(f'Epochs: {epoch + 1}, Loss: {avg_loss}')

In [ ]:
model.eval()

In [ ]:
# Evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move to Gpu
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.size(0)

    correct = correct + (predicted == batch_labels).sum().item()
print(correct/total)

In [ ]:
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    # move to Gpu
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.size(0)

    correct = correct + (predicted == batch_labels).sum().item()
print(correct/total)